# Count each term in the taxonomy

### load generated data from the result dir json files

In [ ]:
n = 30 # number of texts (papers, models etc) per one-month-period
n_long = 30
RESULTS_DIR = "../all_results"

In [ ]:
from pathlib import Path
import json
import re

RESULTS_DIR = Path(RESULTS_DIR)
ARXIV_ID_PATTERN = re.compile(r'^\d{4}\.\d{4,5}(v\d+)?$')

arxiv_paths: dict[str, list[str]] = {}
hf_paths: dict[str, list[str]] = {}
dlw_paths : dict[str, list[str]] = {}
unknown_keys: list[str] = []

for json_path in RESULTS_DIR.glob('*.json'):
    key = json_path.stem
    with json_path.open() as handle:
        payload = json.load(handle)

    path_value = None
    for value in payload.values():
        if isinstance(value, dict):
            try:
                path_value = [*value["arch"], *value["prob"], *value["para"], *value["appl"]]
                path_value = [string.replace("_"," ") for string in path_value] # slight syntax adjustment
            except KeyError as e:
                print(value)
                raise e
            #path_value = value

    if path_value is None:
        continue

    if ARXIV_ID_PATTERN.fullmatch(key):
        arxiv_paths[key] = path_value
    elif "__" in key:#any(char.isalpha() for char in key):
        hf_paths[key] = path_value
    elif key.startswith("dlw_"):
        dlw_paths[key] = path_value
    else:
        unknown_keys.append(key)

print(f'Loaded \n\t{len(arxiv_paths)} arXiv paths,\n\t{len(hf_paths)} HF model paths, \n\t{len(dlw_paths)} dl-weekly paths',)
if unknown_keys:
    print(f'WARNING: {len(unknown_keys)} filenames were not classified: {unknown_keys[:5]}')



### import taxonomy and count occurences of each term

In [7]:
import os, sys
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from llmdap.metadata_schemas.taxonomy_traverser import AI_TAXONOMY

ccs_strings = str(AI_TAXONOMY).split("'")[1::2]
ccs_counts_hf = {s:0 for s in ccs_strings}
ccs_counts_ax = ccs_counts_hf.copy()
ccs_counts_dl = ccs_counts_hf.copy()

for key in hf_paths:
    for term in hf_paths[key]:
        try:
            ccs_counts_hf[term] += 1
        except KeyError:
            if not term == "Other":
                print(term)
                #print(key,term)# hf_paths[key])

for key in arxiv_paths:
    for term in arxiv_paths[key]:
        try:
            ccs_counts_ax[term] += 1
        except KeyError:
            if not term == "Other":
                print(term)
                #print(key, term)#arxiv_paths[key])

for key in dlw_paths:
    for term in dlw_paths[key]:
        try:
            ccs_counts_dl[term] += 1
        except KeyError:
            if not term == "Other":
                print(term)
                #print(key, tldr_paths[key])

In [ ]:
print("Hf\tArxiv\ttldr\timpAi\tDL-w\n")
for term in ccs_strings:
    hfc, axc = ccs_counts_hf[term], ccs_counts_ax[term]
    dlw = ccs_counts_dl[term]

    if  hfc+ axc > 10:
        print(hfc, "\t", axc, "\t", dlw, "\t", term)
    #if imc+ tlc +hfc+ axc == 0:
    #    print(term)



# Find time stamps

In [ ]:
from llmdap.dataset_loader import Arxiv_HF_Newsletters_datasets
ahd = Arxiv_HF_Newsletters_datasets()
ahd.prepare()
hf_df, arx_df, nl_dfs = ahd.sample_subsets(n)
dlw_df = nl_dfs[0]

In [ ]:
import pandas as pd
from llmdap.dataset_loader import Longterm_Datasets
ld = Longterm_Datasets()
ld.prepare()
arx_old, dlw_old = ld.sample_subsets(n_long)
hf_new, arx_new, nl_dfs_new = ahd.sample_subsets(n_long)
dlw_new = nl_dfs_new[0]

arx_long = pd.concat([arx_old, arx_new])
dlw_long = pd.concat([dlw_old, dlw_new])



In [11]:
hf_df = hf_df.set_index("modelId")

# Compile to single results files

In [ ]:
hf_df["date"] = hf_df["createdAt"]
arx_df["date"] = arx_df["submission_date"]
arx_long["date"] = arx_long["submission_date"]


# drop unused cols
hf_df = hf_df.drop(["card", "tags", "pipeline_tag", "bin", "createdAt"], axis=1)
arx_df = arx_df.drop(["title", "abstract", "authors_parsed", "bin", "submission_date"], axis=1)
arx_long = arx_long.drop(["title", "abstract", "authors_parsed", "bin", "submission_date"], axis=1)
dlw_df = dlw_df.drop(["text","bin"], axis=1)
dlw_long = dlw_long.drop(["text","bin"], axis=1)


all_dfs = {
    "arx":arx_df,
    "arx_long":arx_long,
    "hf":hf_df,
    "dlw":dlw_df,
    "dlw_long":dlw_long,
}

# Add predicted tags
for key in all_dfs:
    all_dfs[key]["predicted_tag"] = ""

for modelid in hf_df.index:
    hf_df.loc[modelid, "predicted_tag"]= str(hf_paths[modelid.replace("/","__")])
for paperid in arx_df.index:
    arx_df.loc[paperid, "predicted_tag"]= str(arxiv_paths[paperid])
for paperid in arx_long.index:
    arx_long.loc[paperid, "predicted_tag"]= str(arxiv_paths[paperid])
for text_id in dlw_df.index:
    dlw_df.loc[text_id, "predicted_tag"]= str(dlw_paths[text_id])
for text_id in dlw_long.index:
    dlw_long.loc[text_id, "predicted_tag"]= str(dlw_paths[text_id])

In [ ]:
# save
for key in all_dfs:
    all_dfs[key].to_csv("../output_labels/"+key+"_trends.csv.bz2")